# Aula 0: ambiente funcionando

**Abrindo a Caixa-Preta** · Programação e IA para pesquisa em comunicação

Este notebook confirma que o seu ambiente está pronto para o curso. Ao final dele você terá:

1. rodado código Python no Colab;
2. conversado com um modelo de linguagem usando as suas próprias chaves (NVIDIA e OpenRouter);
3. guardado este arquivo no seu repositório do GitHub, com uma mensagem de commit.

**Como funciona um notebook.** Ele é feito de células. As células de texto, como esta, só explicam. As células de código têm um botão de play à esquerda: clique nele, ou use `Shift + Enter`, para rodar. O resultado aparece logo abaixo.

**Antes de começar:** vá em `Arquivo > Salvar uma cópia no Drive`. Você passa a trabalhar na sua cópia.

**Checkpoint.** Sempre que aparecer a palavra *Checkpoint*, mande no chat da aula um print do que apareceu na sua tela.

## Parte 1. O Colab funciona

Rode a célula abaixo.

In [1]:
print("Olá, caixa-preta")

Olá, caixa-preta


**Checkpoint 1:** print da célula com a frase que apareceu embaixo dela.

## Parte 2. Sua primeira edição

Antes de rodar, leia a célula abaixo e responda mentalmente: o que vai aparecer na tela?

Depois troque os textos entre aspas pelo seu nome e pelo seu tema de pesquisa, e rode.

In [2]:
nome = "escreva seu nome aqui"
tema = "escreva seu tema de pesquisa em uma frase"

print(f"Meu nome é {nome} e eu pesquiso {tema}.")

Meu nome é escreva seu nome aqui e eu pesquiso escreva seu tema de pesquisa em uma frase.


O que você acabou de usar:

- `nome` e `tema` são **variáveis**: etiquetas coladas em um valor.
- O texto entre aspas é uma **string**.
- `print` é uma **função**: ela recebe algo e mostra na tela.
- O `f` antes das aspas permite encaixar variáveis dentro do texto, entre chaves.

**Checkpoint 2:** print da frase com o seu nome e o seu tema.

## Parte 3. Preparação para falar com um modelo

A célula abaixo é mais longa. Você **não precisa entender tudo agora**: na aula 4 a gente abre essa caixa linha por linha. Por enquanto, repare em três coisas ao passar o olho:

- existe uma lista de **provedores**, cada um com um endereço e o nome de um segredo;
- existe uma função chamada `perguntar`, que recebe o provedor e a pergunta;
- em nenhum lugar aparece uma chave escrita. Ela é buscada nos **Secrets** do Colab.

Rode a célula uma vez. Ela só prepara as ferramentas e não envia nada.

In [14]:
from google.colab import userdata
from openai import OpenAI

# Os dois provedores falam o mesmo "idioma" (o formato da OpenAI).
# Só mudam o endereço, a chave e os nomes dos modelos.
PROVEDORES = {
    "nvidia": {
        "endereco": "https://integrate.api.nvidia.com/v1",
        "segredo": "NVIDIA_API_KEY",
        "modelos": ["google/gemma-4-31b-it", "mistralai/mistral-large-4-1t-a52b", "nvidia/nemotron-3.5-lightning-30b-a3b"],
    },
    "openrouter": {
        "endereco": "https://openrouter.ai/api/v1",
        "segredo": "OPENROUTER_API_KEY",
        "modelos": ["openrouter/free"],
    },
}


def perguntar(provedor, pergunta, mostrar=True):
    """Envia uma pergunta a um provedor e devolve o texto da resposta."""
    config = PROVEDORES[provedor]
    segredo = config["segredo"]

    try:
        chave = userdata.get(segredo)
    except Exception as erro:
        if "Access" in type(erro).__name__:
            print(f"O segredo {segredo} existe, mas este notebook não tem acesso a ele.")
            print("No painel da chave, ligue o botão 'Acesso do notebook' e rode de novo.")
        else:
            print(f"Não encontrei o segredo {segredo}.")
            print("Abra o ícone de chave na barra lateral, crie um segredo com esse nome exato e cole a chave no valor.")
        return None

    cliente = OpenAI(base_url=config["endereco"], api_key=chave)
    erros = []
    for modelo in config["modelos"]:
        try:
            resposta = cliente.chat.completions.create(
                model=modelo,
                messages=[{"role": "user", "content": pergunta}],
                max_tokens=400,
            )
            texto = (resposta.choices[0].message.content or "").strip()
            if not texto:
                raise ValueError("O modelo devolveu uma resposta vazia.")
            if mostrar:
                print(f"[{provedor} | modelo: {resposta.model}]")
                print(texto)
            return texto
        except Exception as erro:
            erros.append((modelo, erro))

    for modelo, erro in erros:
        codigo = getattr(erro, "status_code", None)
        print(f"- {modelo}:")
        if codigo in (401, 403):
            print(f"  O provedor recusou a chave ({codigo}). Confira se o valor do segredo {segredo} foi colado inteiro, sem espaços.")
        elif codigo == 429:
            print("  Limite de uso atingido (429). Espere um minuto e rode de novo.")
        elif codigo in (404, 410):
            print(f"  Modelo indisponível ou aposentado ({codigo}). Avise o professor: basta trocar o nome em PROVEDORES.")
        else:
            print("  Não foi possível falar com o provedor. Copie a mensagem abaixo e mande no chat:")
            print(f"  {erro}")
    return None

print("Preparação concluída.")

Preparação concluída.


## Parte 4. Chave da NVIDIA

A chave é uma senha que identifica você no provedor. Ela **nunca** é colada em uma célula de código.

1. No guia de setup, siga o passo 3 para criar a sua chave em `build.nvidia.com`. Ela começa com `nvapi-`.
2. Aqui no Colab, clique no **ícone de chave** na barra lateral esquerda (Secrets).
3. Clique em **Adicionar novo secret**.
4. Em **Nome**, escreva exatamente `NVIDIA_API_KEY`. Em **Valor**, cole a chave.
5. Ligue o botão **Acesso do notebook**.
6. Rode a célula abaixo.

In [15]:
resposta_nvidia = perguntar("nvidia", "Em uma frase curta, o que é um token para um modelo de linguagem?")

[nvidia | modelo: google/gemma-4-31b-it]
Um token é a unidade básica de texto (como uma palavra, parte de uma palavra ou um caractere) que um modelo de linguagem processa e prevê.


**Checkpoint 3:** print da resposta. Confira antes se a chave não aparece em nenhum lugar da tela.

## Parte 5. Chave do OpenRouter

O OpenRouter é o nosso provedor reserva. Se a NVIDIA estiver lenta ou fora do ar em alguma aula, trocamos uma palavra no código e seguimos.

1. No guia de setup, siga o passo 4 para criar a sua chave em `openrouter.ai`. Ela começa com `sk-or-`.
2. No ícone de chave, adicione outro secret com o nome exato `OPENROUTER_API_KEY`.
3. Ligue o **Acesso do notebook** e rode a célula abaixo.

In [5]:
resposta_openrouter = perguntar("openrouter", "Em uma frase curta, o que é um token para um modelo de linguagem?")

[openrouter | modelo: liquid/lfm-3-1.7b:free]
Um token é a menor unidade do texto que o modelo reconhece e manipula.


**Checkpoint 4:** print da resposta.

## Parte 6. Exercício: a mesma pergunta, três respostas

Escreva uma pergunta sobre o **seu tema de pesquisa**, de preferência uma cuja resposta você conhece bem. Depois rode a célula.

Ela envia a mesma pergunta duas vezes para a NVIDIA e uma vez para o OpenRouter.

In [16]:
minha_pergunta = "escreva aqui uma pergunta sobre o seu tema"

print("NVIDIA, primeira vez")
a = perguntar("nvidia", minha_pergunta)

print()
print("NVIDIA, segunda vez")
b = perguntar("nvidia", minha_pergunta)

print()
print("OpenRouter")
c = perguntar("openrouter", minha_pergunta)

NVIDIA, primeira vez
[nvidia | modelo: google/gemma-4-31b-it]
Como eu sou uma Inteligência Artificial generalista, meu "tema" é, na verdade, o processamento de informações e a linguagem humana. 

Para testar minhas capacidades ou aprofundar nossa conversa, aqui está uma pergunta instigante que você pode me fazer:

> **"Se você pudesse criar uma nova habilidade para si mesma que não dependesse apenas de processar dados existentes, mas de 'sentir' ou 'experimentar' o mundo real, qual seria essa habilidade e como ela mudaria a forma como você me ajuda?"**

---

**Ou, se você preferir algo mais técnico/prático, tente esta:**

> **"Quais são as limitações éticas e técnicas que impedem que você tenha consciência própria, e como você diferencia 'simular empatia' de 'sentir empatia'?"**

Qual delas você prefere responder (ou me perguntar)?

NVIDIA, segunda vez
[nvidia | modelo: google/gemma-4-31b-it]
Como eu sou uma Inteligência Artificial generalista, meu "tema" é, na verdade, o processamento

Dê dois cliques nesta célula e responda, em uma linha cada:

1. As duas respostas da NVIDIA foram iguais?
2. Alguma das três respostas tem um erro ou uma afirmação que você precisaria conferir? Qual?
3. Se fosse citar uma delas na sua pesquisa, o que você teria de registrar para outra pessoa conseguir repetir o teste?

*Suas respostas:*

1.
2.
3.

**Checkpoint 5:** escreva no chat uma diferença que você notou entre as respostas.

## Parte 7. Guardar no GitHub

Agora este notebook vira o primeiro arquivo do seu repositório.

1. Use `Ctrl + F` (ou `Cmd + F`) e procure por `nvapi` e por `sk-or`. Não deve aparecer nenhuma chave. Se aparecer, apague, gere uma chave nova no provedor e avise no chat.
2. Vá em `Arquivo > Fazer download > Fazer download do .ipynb`.
3. Abra o seu repositório no GitHub e clique em `Add file > Upload files`.
4. Arraste o arquivo baixado.
5. Em **Commit changes**, escreva uma mensagem que diga o que mudou. Exemplo: `Adiciona notebook da aula 0 com teste das chaves`.
6. Clique em **Commit changes**.

**Checkpoint 6:** print da página do repositório mostrando o arquivo e a mensagem de commit.

## Pronto

Se os seis checkpoints foram enviados, o seu ambiente está completo para o curso.

| Item | Feito |
| --- | --- |
| Colab roda código | |
| Chave da NVIDIA responde | |
| Chave do OpenRouter responde | |
| Notebook guardado no GitHub com mensagem de commit | |

A lição de casa está no arquivo `licao-de-casa.md`. Leva de 20 a 30 minutos.